In [64]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset,DataLoader

In [65]:
train_data_raw = pd.read_csv('train.csv')
test_data_raw = pd.read_csv('test.csv')
train_data_raw

,id,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_85,feat_86,feat_87,feat_88,feat_89,feat_90,feat_91,feat_92,feat_93,target
0,1,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,Class_1
1,2,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,Class_1
2,3,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,Class_1
3,4,1,0,0,1,6,1,5,0,0,...,0,1,2,0,0,0,0,0,0,Class_1
4,5,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,Class_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61873,61874,1,0,0,1,1,0,0,0,0,...,1,0,0,0,0,0,0,2,0,Class_9
61874,61875,4,0,0,0,0,0,0,0,0,...,0,2,0,0,2,0,0,1,0,Class_9
61875,61876,0,0,0,0,0,0,0,3,1,...,0,3,1,0,0,0,0,0,0,Class_9
61876,61877,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,3,10,0,Class_9


In [66]:
train_x = train_data_raw.iloc[:, 1:-1]
train_x = train_x.to_numpy().astype(np.float32)

train_y = train_data_raw.iloc[:, -1]
y = train_y.str.replace("Class_", "", regex=False).astype(int)
train_y = y.to_numpy().astype(int)
train_y -= 1

In [67]:
BATCH_SIZE = 64

class TrainData(Dataset):
    def __init__(self,train_x,train_y):
        self.x_data = torch.tensor(train_x,dtype=torch.float32)
        self.y_data = torch.tensor(train_y,dtype=torch.long)

    def __getitem__(self, index):
        return self.x_data[index],self.y_data[index]
    
    def __len__(self):
        return self.x_data.shape[0]

train_data = TrainData(train_x,train_y)
train_load = DataLoader(train_data,shuffle=True,batch_size=BATCH_SIZE)
test_load = DataLoader(train_data,shuffle=False,batch_size=BATCH_SIZE)

In [68]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = torch.nn.Linear(93,256,bias=True)
        self.l2 = torch.nn.Linear(256,128)
        self.l3 = torch.nn.Linear(128,64)
        self.l4 = torch.nn.Linear(64,9)
        self.Relu = torch.nn.ReLU()
        self.drop = torch.nn.Dropout(0.2)

    def forward(self,x):
        x = self.Relu(self.l1(x))
        x = self.drop(x)
        x = self.Relu(self.l2(x))
        x = self.drop(x)
        x = self.Relu(self.l3(x))
        x = self.drop(x)
        return self.l4(x)
    
model = NeuralNetwork().to('cuda')

In [69]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.01,momentum=0.5)

In [70]:
def train(epoch):
    running_loss = 0
    for batch_idx,data in enumerate(train_load,0):
        inputs,target = data
        inputs = inputs.to('cuda')
        target = target.to('cuda')
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs,target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch , batch_idx + 1, running_loss / 300))
            running_loss = 0.0

In [71]:
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_load:
            images, labels = data
            images = images.to('cuda')
            labels = labels.to('cuda')
            outputs = model(images)
            _,predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print('Accuracy on test set: %d %%' % (100 * correct / total))

In [72]:
for epoch in range(1,16):
    train(epoch)
    test()

[1,   300] loss: 1.503
[1,   600] loss: 0.941
[1,   900] loss: 0.841
Accuracy on test set: 72 %
[2,   300] loss: 0.785
[2,   600] loss: 0.763
[2,   900] loss: 0.743
Accuracy on test set: 73 %
[3,   300] loss: 0.718
[3,   600] loss: 0.711
[3,   900] loss: 0.695
Accuracy on test set: 74 %
[4,   300] loss: 0.682
[4,   600] loss: 0.675
[4,   900] loss: 0.673
Accuracy on test set: 75 %
[5,   300] loss: 0.656
[5,   600] loss: 0.660
[5,   900] loss: 0.654
Accuracy on test set: 75 %
[6,   300] loss: 0.640
[6,   600] loss: 0.644
[6,   900] loss: 0.640
Accuracy on test set: 76 %
[7,   300] loss: 0.617
[7,   600] loss: 0.631
[7,   900] loss: 0.627
Accuracy on test set: 76 %
[8,   300] loss: 0.608
[8,   600] loss: 0.609
[8,   900] loss: 0.608
Accuracy on test set: 76 %
[9,   300] loss: 0.605
[9,   600] loss: 0.593
[9,   900] loss: 0.595
Accuracy on test set: 77 %
[10,   300] loss: 0.588
[10,   600] loss: 0.585
[10,   900] loss: 0.592
Accuracy on test set: 77 %
[11,   300] loss: 0.576
[11,   600] l

In [73]:
# Predict on test.csv and write submission
device = next(model.parameters()).device
test_x = test_data_raw.iloc[:, 1:].to_numpy().astype(np.float32)
test_tensor = torch.tensor(test_x, dtype=torch.float32)
test_ds = torch.utils.data.TensorDataset(test_tensor)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

model.eval()
preds = []
with torch.no_grad():
    for (x_batch,) in test_loader:
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        preds.append(torch.argmax(outputs, dim=1).cpu().numpy())
pred = np.concatenate(preds)

pred_onehot = np.eye(9, dtype=np.int64)[pred]
sub = pd.read_csv('sampleSubmission.csv')
sub.iloc[:, 1:] = pred_onehot
sub.to_csv('submission.csv', index=False)
sub.head()

,id,Class_1,Class_2,Class_3,Class_4,Class_5,Class_6,Class_7,Class_8,Class_9
0,1,0,0,1,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,1,0
2,3,0,0,0,0,0,1,0,0,0
3,4,0,1,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0,0,1
